In [0]:
%pip install databricks-feature-engineering
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow

model = mlflow.sklearn.load_model("runs:/8cf694c973c1457d86e9cf5013609090/model")



In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

def import_query(path):
    with open(path) as f:
        return f.read()

feature_lookups = [
    FeatureLookup(table_name="feature_store.credit_score.fs_cadastral", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_temporal", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_historico_financeiro", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_renda", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_funcionarios", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_historico_pagamentos", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"])
]

fe = FeatureEngineeringClient()

query = """
    SELECT
        DATA_REF,
        ID_CLIENTE,
        ID_DOCUMENTO
    FROM feature_store.credit_score.fs_cadastral
    WHERE DATA_REF = (
        SELECT MAX(DATA_REF)
        FROM feature_store.credit_score.fs_cadastral
    )
"""

df = spark.sql(query)


predict_set = fe.create_training_set(df=df, feature_lookups=feature_lookups, label=None)
df_predict = predict_set.load_df().toPandas()

In [0]:
df_predict.head()

In [0]:
df_predict["prediction"] = model.predict(df_predict)
df_predict["probabilities"] = model.predict_proba(df_predict)[:, 1]

In [0]:
df_predict.head()

In [0]:
pagamentos_df = spark.sql("""
    SELECT ID_DOCUMENTO, DATA_PAGAMENTO, DATA_VENCIMENTO
    FROM credit_score.data.pagamentos
""")

df_predict_spark = spark.createDataFrame(df_predict)
df_predict_joined = df_predict_spark.join(pagamentos_df, on="ID_DOCUMENTO", how="left")
df_predict = df_predict_joined.toPandas()

In [0]:
import pandas as pd

df_predict["inad_real"] = ((pd.to_datetime(df_predict["DATA_PAGAMENTO"]) - pd.to_datetime(df_predict["DATA_VENCIMENTO"])).dt.days >= 5).astype(int)

In [0]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(df_predict["inad_real"], df_predict["probabilities"])
auc